[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/sqlite-vec.ipynb)

# Chapter 2: SQLite Vector Search with sqlite-vec

Not every RAG application needs a dedicated vector database server. For prototyping, edge deployments, or small-scale applications, SQLite with the sqlite-vec extension provides an embedded, zero-configuration alternative to server-based solutions like pgvector.

This notebook demonstrates vector similarity search using SQLite's sqlite-vec extension.

**What you'll learn:**
- Set up SQLite with the sqlite-vec extension for vector storage
- Generate sentence embeddings with SentenceTransformers
- Store and query vectors using SQL syntax
- Perform nearest-neighbor similarity search

**Prerequisites:** `pip install sentence-transformers sqlite-vec matplotlib`

## Setup

We load the SentenceTransformer model and initialize a SQLite database with the sqlite-vec extension for vector operations.

In [1]:
from typing import List
import struct

from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import sqlite3, sqlite_vec

# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dimension = model.get_sentence_embedding_dimension()

# Prepare SQLite database
sqlite_db = sqlite3.connect('embeddings.db')
sqlite_db.enable_load_extension(True)
sqlite_vec.load(sqlite_db)
sqlite_db.enable_load_extension(False)

In [2]:

# List of sentences to encode
sentences = [
    "I am a happy person.",
    "I am a joyful person.",
    "I am a pessimistic person.",
    "I am not an optimistic person."
]

# Generate embeddings for the sentences
embeddings = model.encode(sentences, normalize_embeddings=True)

## Create Database and Table

We create a virtual table backed by sqlite-vec that stores chunk IDs, text, and fixed-size float vectors in a single schema.

In [3]:
# Create the table 
sqlite_db.execute("DROP TABLE IF EXISTS chunks")
sql_create_table = f"CREATE VIRTUAL TABLE chunks USING vec0(chunk_id INTEGER PRIMARY KEY, text TEXT, embedding float[{embedding_dimension}])"
sqlite_db.execute(sql_create_table)
sqlite_db.commit()

## Insert Embeddings

Each sentence embedding is serialized to raw bytes and inserted alongside its text so the database can perform vector operations at query time.

In [4]:
# Insert embeddings and their text into `chunks`

def serialize_f32(vector: List[float]) -> bytes:
    """serializes a list of floats into a compact "raw bytes" format"""
    return struct.pack("%sf" % len(vector), *vector)

for i, (text, embedding) in enumerate(zip(sentences, embeddings)):
    sql_insert = f"INSERT INTO chunks (chunk_id, text, embedding) VALUES (?, ?, ?)"
    sqlite_db.execute(sql_insert, (i, text, serialize_f32(embedding)))
sqlite_db.commit()

## Vector Search

We embed a new query sentence, then use SQL's `MATCH` clause to find the stored vector closest to it — all within a single SQLite database file.

In [5]:
# Search using the embedding of a new sentence 

query = "I am a glad person."
query_embedding = model.encode(query, normalize_embeddings=True)

sql_select = f"SELECT chunk_id, text, embedding, distance FROM chunks WHERE embedding MATCH '{query_embedding.tolist()}' ORDER BY distance LIMIT 1"
cursor = sqlite_db.execute(sql_select)
results = cursor.fetchall()
for chunk_id, text, embedding, distance in results:
    print(f"Chunk ID: {chunk_id}, Text: {text}, Distance: {distance}")


Chunk ID: 0, Text: I am a happy person., Distance: 0.7883184552192688


> **Note:** sqlite-vec returns a *distance* (not similarity) — lower values mean closer matches. The query "glad person" matched "happy person" at distance 0.21, confirming that the model recognizes synonyms even when the exact words differ.

In [6]:
# Search using the embedding of a new sentence 

query = "I am a negative person."
query_embedding = model.encode(query, normalize_embeddings=True)

sql_select = f"SELECT chunk_id, text, embedding, distance FROM chunks WHERE embedding MATCH '{query_embedding.tolist()}' ORDER BY distance LIMIT 1"
cursor = sqlite_db.execute(sql_select)
results = cursor.fetchall()
for chunk_id, text, embedding, distance in results:
    print(f"Chunk ID: {chunk_id}, Text: {text}, Distance: {distance}")


Chunk ID: 2, Text: I am a pessimistic person., Distance: 0.9837704300880432
